# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[ 0.36756007  0.42705883  0.39265849  0.21047761  0.50669835]
 [-0.25153978  0.25409975  0.43679212 -0.21209973 -0.09036974]
 [ 0.03550231 -0.02123978 -0.44910867  0.29698612  0.77894011]
 [ 0.94990816  0.49288677  0.27125369 -0.35726428 -0.4032978 ]
 [-0.64443097 -0.40577265 -0.17733091  0.80565374 -0.31567676]
 [-0.23524478 -0.47819751  0.08128759  0.88364792 -0.36456692]
 [ 0.8356463  -0.9407115  -0.01474058  0.73093938  0.10289761]
 [ 0.01801531  0.23183813  0.3791961   0.02017618 -0.10765023]
 [ 0.44529639 -0.92390029  0.91718731 -0.32884594  0.60167439]
 [ 0.53444482 -0.36304533 -0.54655775 -0.5490287   0.53264142]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a1', 'a2', 'a1', 'a2', 'a1', 'a2', 'a2', 'a1', 'a2', 'a2']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [0, 0, 0, 1, 1, 0, 0, 1, 0, 1]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:36,  1.10s/it]

SVI:   3%|▎         | 1/34 [00:01<00:36,  1.10s/it, loss=2364.8213]

SVI:   6%|▌         | 2/34 [00:01<00:35,  1.10s/it, loss=2912.0432]

SVI:   9%|▉         | 3/34 [00:01<00:34,  1.10s/it, loss=2573.3425]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.21it/s, loss=3582.6191]

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.21it/s, loss=2165.9001]

SVI:  97%|█████████▋| 33/34 [00:00<00:00,  1.21it/s, loss=2324.4355]

SVI: 100%|██████████| 34/34 [00:01<00:00, 24.16it/s, loss=2324.4355]

SVI: 100%|██████████| 34/34 [00:01<00:00, 24.16it/s, loss=3360.4395]